In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q xgboost

import os
import glob
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, precision_score, recall_score, f1_score)

PROJECT_DIR = '/content/drive/MyDrive/man-bot-detection'
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'notebooks/results')
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)

RANDOM_STATE = 42

In [ ]:
FEATURE_COLS_BEHAV = ['mean_speed', 'max_speed', 'std_speed', 'path_length',
                       'num_stops', 'mean_angle', 'num_clicks', 'num_points']

behav_csv = f'{PROJECT_DIR}/data/processed/behavioral_features_windowed.csv'
df_behav = pd.read_csv(behav_csv)

X_behav = df_behav[FEATURE_COLS_BEHAV]
y_behav = df_behav['label']

X_behav_train, X_behav_test, y_behav_train, y_behav_test = train_test_split(
    X_behav, y_behav, test_size=0.2, stratify=y_behav, random_state=RANDOM_STATE
)

rf_behav = joblib.load(f'{PROJECT_DIR}/models/behavioral_rf_300.pkl')
proba_behav_test = rf_behav.predict_proba(X_behav_test)[:, list(rf_behav.classes_).index('bot')]

behav_scores_by_class = {
    'bot': proba_behav_test[(y_behav_test == 'bot').values],
    'human': proba_behav_test[(y_behav_test == 'human').values],
}
print(len(X_behav_test), sum(y_behav_test=='bot'), sum(y_behav_test=='human'))
print(behav_scores_by_class['bot'].mean())
print(behav_scores_by_class['human'].mean())

In [ ]:
DATA_BASE = os.path.join(PROJECT_DIR, 'data/raw')
CIC_PATH = os.path.join(DATA_BASE, 'cic_ids_2017')
CTU_PATH = os.path.join(DATA_BASE, 'ctu13')


def load_cic():
    files = glob.glob(os.path.join(CIC_PATH, '*.csv'))
    df = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
    if len(df) > 200000:
        df = df.sample(n=200000, random_state=RANDOM_STATE)
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    y = (df[' Label'].astype(str).str.strip() != 'BENIGN').astype(int)
    return df.reset_index(drop=True), y.reset_index(drop=True)


def load_ctu():
    attack_file = os.path.join(CTU_PATH, 'CTU13_Attack_Traffic.csv')
    normal_file = os.path.join(CTU_PATH, 'CTU13_Normal_Traffic.csv')
    df_attack = pd.read_csv(attack_file, low_memory=False)
    df_normal = pd.read_csv(normal_file, low_memory=False)

    raw_label_variants = ['Label', 'LABEL', 'Label_original']
    df_attack = df_attack.drop(columns=[c for c in raw_label_variants if c in df_attack.columns], errors='ignore')
    df_normal = df_normal.drop(columns=[c for c in raw_label_variants if c in df_normal.columns], errors='ignore')
    common_cols = df_attack.columns.intersection(df_normal.columns)
    df_attack, df_normal = df_attack[common_cols].copy(), df_normal[common_cols].copy()
    df_attack['label'] = 1
    df_normal['label'] = 0
    df = pd.concat([df_attack, df_normal], ignore_index=True)
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    y = df['label']
    X = df.drop(columns=['label'])
    return X.reset_index(drop=True), y.reset_index(drop=True)


FEATURE_MAP_CIC_CTU = {
    'flow_duration':        {'CIC': ' Flow Duration',               'CTU': 'Flow Duration'},
    'total_fwd_packets':     {'CIC': ' Total Fwd Packets',           'CTU': 'Tot Fwd Pkts'},
    'total_bwd_packets':     {'CIC': ' Total Backward Packets',      'CTU': 'Tot Bwd Pkts'},
    'total_fwd_bytes':       {'CIC': 'Total Length of Fwd Packets',  'CTU': 'TotLen Fwd Pkts'},
    'total_bwd_bytes':       {'CIC': ' Total Length of Bwd Packets', 'CTU': 'TotLen Bwd Pkts'},
    'fwd_pkt_len_max':       {'CIC': ' Fwd Packet Length Max',       'CTU': 'Fwd Pkt Len Max'},
    'fwd_pkt_len_min':       {'CIC': ' Fwd Packet Length Min',       'CTU': 'Fwd Pkt Len Min'},
    'fwd_pkt_len_mean':      {'CIC': ' Fwd Packet Length Mean',      'CTU': 'Fwd Pkt Len Mean'},
    'fwd_pkt_len_std':       {'CIC': ' Fwd Packet Length Std',       'CTU': 'Fwd Pkt Len Std'},
    'bwd_pkt_len_max':       {'CIC': 'Bwd Packet Length Max',        'CTU': 'Bwd Pkt Len Max'},
    'bwd_pkt_len_min':       {'CIC': ' Bwd Packet Length Min',       'CTU': 'Bwd Pkt Len Min'},
    'bwd_pkt_len_mean':      {'CIC': ' Bwd Packet Length Mean',      'CTU': 'Bwd Pkt Len Mean'},
    'bwd_pkt_len_std':       {'CIC': ' Bwd Packet Length Std',       'CTU': 'Bwd Pkt Len Std'},
    'flow_bytes_per_s':      {'CIC': 'Flow Bytes/s',                 'CTU': 'Flow Byts/s'},
    'flow_packets_per_s':    {'CIC': ' Flow Packets/s',              'CTU': 'Flow Pkts/s'},
    'flow_iat_mean':         {'CIC': ' Flow IAT Mean',               'CTU': 'Flow IAT Mean'},
    'flow_iat_std':          {'CIC': ' Flow IAT Std',                'CTU': 'Flow IAT Std'},
    'flow_iat_max':          {'CIC': ' Flow IAT Max',                'CTU': 'Flow IAT Max'},
    'flow_iat_min':          {'CIC': ' Flow IAT Min',                'CTU': 'Flow IAT Min'},
    'fwd_iat_total':         {'CIC': 'Fwd IAT Total',                'CTU': 'Fwd IAT Tot'},
    'fwd_iat_mean':          {'CIC': ' Fwd IAT Mean',                'CTU': 'Fwd IAT Mean'},
    'fwd_iat_std':           {'CIC': ' Fwd IAT Std',                 'CTU': 'Fwd IAT Std'},
    'fwd_iat_max':           {'CIC': ' Fwd IAT Max',                 'CTU': 'Fwd IAT Max'},
    'fwd_iat_min':           {'CIC': ' Fwd IAT Min',                 'CTU': 'Fwd IAT Min'},
    'bwd_iat_total':         {'CIC': 'Bwd IAT Total',                'CTU': 'Bwd IAT Tot'},
    'bwd_iat_mean':          {'CIC': ' Bwd IAT Mean',                'CTU': 'Bwd IAT Mean'},
    'bwd_iat_std':           {'CIC': ' Bwd IAT Std',                 'CTU': 'Bwd IAT Std'},
    'bwd_iat_max':           {'CIC': ' Bwd IAT Max',                 'CTU': 'Bwd IAT Max'},
    'bwd_iat_min':           {'CIC': ' Bwd IAT Min',                 'CTU': 'Bwd IAT Min'},
    'bwd_psh_flags':         {'CIC': ' Bwd PSH Flags',               'CTU': 'Bwd PSH Flags'},
    'fwd_header_len':        {'CIC': ' Fwd Header Length',           'CTU': 'Fwd Header Len'},
    'bwd_header_len':        {'CIC': ' Bwd Header Length',           'CTU': 'Bwd Header Len'},
    'fwd_packets_per_s':     {'CIC': 'Fwd Packets/s',                'CTU': 'Fwd Pkts/s'},
    'bwd_packets_per_s':     {'CIC': ' Bwd Packets/s',               'CTU': 'Bwd Pkts/s'},
    'min_packet_len':        {'CIC': ' Min Packet Length',           'CTU': 'Pkt Len Min'},
    'max_packet_len':        {'CIC': ' Max Packet Length',           'CTU': 'Pkt Len Max'},
    'packet_len_mean':       {'CIC': ' Packet Length Mean',          'CTU': 'Pkt Len Mean'},
    'packet_len_std':        {'CIC': ' Packet Length Std',           'CTU': 'Pkt Len Std'},
    'packet_len_var':        {'CIC': ' Packet Length Variance',      'CTU': 'Pkt Len Var'},
    'fin_flag_count':        {'CIC': 'FIN Flag Count',               'CTU': 'FIN Flag Cnt'},
    'syn_flag_count':        {'CIC': ' SYN Flag Count',              'CTU': 'SYN Flag Cnt'},
    'rst_flag_count':        {'CIC': ' RST Flag Count',              'CTU': 'RST Flag Cnt'},
    'ack_flag_count':        {'CIC': ' ACK Flag Count',              'CTU': 'ACK Flag Cnt'},
    'down_up_ratio':         {'CIC': ' Down/Up Ratio',               'CTU': 'Down/Up Ratio'},
    'avg_packet_size':       {'CIC': ' Average Packet Size',         'CTU': 'Pkt Size Avg'},
    'avg_fwd_segment_size':  {'CIC': ' Avg Fwd Segment Size',        'CTU': 'Fwd Seg Size Avg'},
    'avg_bwd_segment_size':  {'CIC': ' Avg Bwd Segment Size',        'CTU': 'Bwd Seg Size Avg'},
    'init_win_bytes_bwd':    {'CIC': ' Init_Win_bytes_backward',     'CTU': 'Init Bwd Win Byts'},
    'act_data_pkt_fwd':      {'CIC': ' act_data_pkt_fwd',            'CTU': 'Fwd Act Data Pkts'},
    'active_mean':           {'CIC': 'Active Mean',                  'CTU': 'Active Mean'},
    'active_std':            {'CIC': ' Active Std',                  'CTU': 'Active Std'},
    'active_max':            {'CIC': ' Active Max',                  'CTU': 'Active Max'},
    'active_min':            {'CIC': ' Active Min',                  'CTU': 'Active Min'},
    'idle_mean':             {'CIC': 'Idle Mean',                    'CTU': 'Idle Mean'},
    'idle_std':              {'CIC': ' Idle Std',                    'CTU': 'Idle Std'},
    'idle_max':              {'CIC': ' Idle Max',                    'CTU': 'Idle Max'},
    'idle_min':              {'CIC': ' Idle Min',                    'CTU': 'Idle Min'},
}


def build_common_features(feature_map, cic_df, ctu_df):
    used_keys, cic_cols, ctu_cols = [], [], []
    for key, mapping in feature_map.items():
        if mapping['CIC'] in cic_df.columns and mapping['CTU'] in ctu_df.columns:
            used_keys.append(key)
            cic_cols.append(mapping['CIC'])
            ctu_cols.append(mapping['CTU'])
    X_cic_al = cic_df[cic_cols].copy(); X_cic_al.columns = used_keys
    X_ctu_al = ctu_df[ctu_cols].copy(); X_ctu_al.columns = used_keys
    print(len(used_keys))
    return X_cic_al, X_ctu_al

In [ ]:
X_cic_raw, y_cic = load_cic()
X_ctu_raw, y_ctu = load_ctu()

X_cic_common, X_ctu_common = build_common_features(FEATURE_MAP_CIC_CTU, X_cic_raw, X_ctu_raw)

scaler_shift = StandardScaler()
X_cic_scaled = scaler_shift.fit_transform(X_cic_common)
X_ctu_scaled = scaler_shift.transform(X_ctu_common)

rf_shift = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                   class_weight='balanced', n_jobs=-1)
rf_shift.fit(X_cic_scaled, y_cic)

proba_net_shift = rf_shift.predict_proba(X_ctu_scaled)[:, 1]

print(len(y_cic), len(y_ctu))
print(proba_net_shift[(y_ctu==1).values].mean())
print(proba_net_shift[(y_ctu==0).values].mean())
print(f1_score(y_ctu, (proba_net_shift>=0.5).astype(int)))

net_shift_scores_by_class = {
    'bot': proba_net_shift[(y_ctu == 1).values],
    'human': proba_net_shift[(y_ctu == 0).values],
}

In [ ]:
N_SIM_PER_CLASS = 5000
rng = np.random.default_rng(RANDOM_STATE)

sim_rows = []
for cls, true_label in [('bot', 1), ('human', 0)]:
    behav_pool = behav_scores_by_class[cls]
    net_pool = net_shift_scores_by_class[cls]
    sampled_behav = rng.choice(behav_pool, size=N_SIM_PER_CLASS, replace=True)
    sampled_net = rng.choice(net_pool, size=N_SIM_PER_CLASS, replace=True)
    for sb, sn in zip(sampled_behav, sampled_net):
        sim_rows.append({'score_behavioral': sb, 'score_network_shift': sn, 'is_bot': true_label})

df_sim = pd.DataFrame(sim_rows)
print(len(df_sim))
print(df_sim.groupby('is_bot')[['score_behavioral', 'score_network_shift']].mean())

In [ ]:
X_sim = df_sim[['score_behavioral', 'score_network_shift']]
y_sim = df_sim['is_bot']

X_sim_train, X_sim_test, y_sim_train, y_sim_test = train_test_split(
    X_sim, y_sim, test_size=0.3, stratify=y_sim, random_state=RANDOM_STATE
)

results_fusion = []


def eval_strategy(name, y_true, y_pred_binary):
    acc = accuracy_score(y_true, y_pred_binary)
    prec = precision_score(y_true, y_pred_binary, zero_division=0)
    rec = recall_score(y_true, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true, y_pred_binary, zero_division=0)
    results_fusion.append({'strategy': name, 'accuracy': acc, 'precision': prec,
                            'recall': rec, 'f1': f1})
    print(name, acc, prec, rec, f1)


eval_strategy("behavioral", y_sim_test,
              (X_sim_test['score_behavioral'] >= 0.5).astype(int))
eval_strategy("network_shift", y_sim_test,
              (X_sim_test['score_network_shift'] >= 0.5).astype(int))

avg_score = (X_sim_test['score_behavioral'] + X_sim_test['score_network_shift']) / 2
eval_strategy("average", y_sim_test, (avg_score >= 0.5).astype(int))

meta_model = LogisticRegression(random_state=RANDOM_STATE)
meta_model.fit(X_sim_train, y_sim_train)
y_pred_meta = meta_model.predict(X_sim_test)
eval_strategy("meta", y_sim_test, y_pred_meta)

print(meta_model.coef_[0][0], meta_model.coef_[0][1], meta_model.intercept_[0])

df_results_fusion = pd.DataFrame(results_fusion)
df_results_fusion.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'fusion_domain_shift_comparison.csv'), index=False)

In [ ]:
plt.figure(figsize=(11, 6))
df_plot = df_results_fusion.melt(id_vars='strategy', value_vars=['accuracy', 'precision', 'recall', 'f1'],
                                  var_name='metric', value_name='value')
ax = sns.barplot(data=df_plot, x='strategy', y='value', hue='metric')
plt.title('fusion')
plt.ylabel('value')
plt.xlabel('')
plt.ylim(0, 1.05)
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'fusion_domain_shift_comparison.png'), dpi=300)
plt.show()

In [ ]:
only_net_shift_f1 = df_results_fusion.loc[
    df_results_fusion['strategy'].str.contains('network_shift'), 'f1'
].iloc[0]
best_row = df_results_fusion.loc[df_results_fusion['f1'].idxmax()]
improvement = best_row['f1'] - only_net_shift_f1

print(only_net_shift_f1)
print(best_row['strategy'])
print(best_row['f1'])
print(improvement)